In [2]:
import polars as pl

ruta = "/Users/macbook/ProyectosLocales/PrecioLuz/datos/precio_pvpc.csv"

df = pl.read_csv(ruta)

# arreglar 24:00
df = df.with_columns(
    pl.when(pl.col("Timestamp").str.contains("T24:00:00"))
    .then(pl.col("Timestamp").str.replace("T24:00:00", "T00:00:00"))
    .otherwise(pl.col("Timestamp"))
    .alias("ts")
)

# pasar a datetime
df = df.with_columns(
    pl.col("ts").str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%S").alias("dt")
)

df = df.with_columns(
    pl.when(pl.col("Timestamp").str.contains("T24:00:00"))
    .then(pl.col("dt") + pl.duration(days=1))
    .otherwise(pl.col("dt"))
    .alias("dt")
)

# diario
df = df.select(
    pl.col("dt").dt.date().alias("date"),
    pl.col("PVPC").cast(pl.Float64).alias("pvpc")
)

df = df.group_by("date").agg(
    pl.col("pvpc").mean()
).sort("date")

# limpiar
nulos = df.null_count()
dup = df.height - df.unique(subset=["date"]).height

df = df.drop_nulls().unique(subset=["date"])

# info básica
print("Filas:", df.height)
print("Rango:", df["date"].min(), df["date"].max())

print(df.select([
    pl.col("pvpc").min().alias("min"),
    pl.col("pvpc").max().alias("max"),
    pl.col("pvpc").mean().alias("media")
]))

print("Nulos:", nulos)
print("Duplicados:", dup)

# fechas faltantes
fechas = pl.date_range(
    pl.date(2015,1,1),
    pl.date(2025,12,31),
    "1d",
    eager=True
)

faltan = pl.DataFrame({"date": fechas}).join(df, on="date", how="anti")

print("Faltan:", faltan.height)

# guardar
df.write_csv("/Users/macbook/ProyectosLocales/PrecioLuz/datos/pvpc_total.csv")

Filas: 4018
Rango: 2015-01-01 2025-12-31
shape: (1, 3)
┌──────────┬──────────┬──────────┐
│ min      ┆ max      ┆ media    │
│ ---      ┆ ---      ┆ ---      │
│ f64      ┆ f64      ┆ f64      │
╞══════════╪══════════╪══════════╡
│ 0.013707 ┆ 0.669024 ┆ 0.103493 │
└──────────┴──────────┴──────────┘
Nulos: shape: (1, 2)
┌──────┬──────┐
│ date ┆ pvpc │
│ ---  ┆ ---  │
│ u32  ┆ u32  │
╞══════╪══════╡
│ 0    ┆ 0    │
└──────┴──────┘
Duplicados: 0
Faltan: 0
